# Consumer Kafka con Spark Structured Streaming

Cuaderno base para consumir el topic `orden-eventos` desde Spark y parsear el JSON
con un esquema común que funcione tanto si el productor fue hecho en **Java** como en **Python**.

## Objetivo
1. Conectarse a Kafka
2. Leer mensajes del topic `orden-eventos`
3. Convertir `value` a texto
4. Parsear el JSON
5. Filtrar eventos válidos
6. Mostrar en consola los eventos con `total > 100`

## Datos del entorno
- Kafka broker interno: `ec-kafka:9092`
- Topic: `orden-eventos`
- Spark: paquete Kafka para Spark SQL

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KafkaSparkOrdenes") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1"
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d1158982-5fad-42e5-8091-0f26b432c932;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#

In [2]:
print("Spark:", spark.version)
print("Scala JVM:", spark.sparkContext._jvm.scala.util.Properties.versionNumberString())

Spark: 4.1.1
Scala JVM: 2.13.17


## Paso 1. Leer el stream desde Kafka

Aquí todavía no parseamos el JSON. Solo leemos los mensajes crudos del topic.

In [3]:
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "ec-kafka:9092") \
    .option("subscribe", "orden-eventos") \
    .option("startingOffsets", "earliest") \
    .load()

In [4]:
df_kafka.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## Paso 2. Convertir `value` a texto (Saltar al paso 3)

Kafka entrega `key` y `value` en binario. Por eso primero convertimos `value` a `STRING`.

In [5]:
df_texto = df_kafka.selectExpr("CAST(value AS STRING) AS mensaje")
df_texto.printSchema()

root
 |-- mensaje: string (nullable = true)



In [6]:
# Vista rápida del mensaje en texto
# Si quieres depurar primero sin parsear JSON, ejecuta esta celda.
query_texto = df_texto.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("truncate", False) \
    .start()

26/04/16 01:34:51 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-b6bab6fb-46e4-4fee-ad3c-b4584789526a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/16 01:34:51 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+---------------------------------------------------------------------------+
|mensaje                                                                    |
+---------------------------------------------------------------------------+
|hi                                                                         |
|orden 1                                                                    |
|{"tipoEvento":"orden.creada","ordenId":1,"estado":"PENDIENTE"}             |
|{"tipoEvento":"orden.creada","ordenId":2,"estado":"PENDIENTE"}             |
|{"tipoEvento":"orden.creada","ordenId":3,"estado":"PENDIENTE"}             |
|{"tipoEvento":"orden.creada","ordenId":4,"estado":"PENDIENTE"}             |
|{"tipoEvento":"orden.creada","ordenId":5,"estado":"PENDIENTE"}             |
|{"tipoEvento":"orden.creada","ordenId":6,"estado":"PENDIENTE"}             |
|orden a                                     

Cuando termines de ver mensajes crudos, detén la consulta antes de seguir:

In [7]:
query_texto.stop()

26/04/16 01:35:07 WARN DAGScheduler: Failed to cancel job group d1d054d2-6614-4048-8431-f678a903fd5e. Cannot find active jobs for it.
26/04/16 01:35:07 WARN DAGScheduler: Failed to cancel job group d1d054d2-6614-4048-8431-f678a903fd5e. Cannot find active jobs for it.


## Paso 3. Definir el esquema común del evento

Este contrato permite que Spark interprete igual los mensajes vengan de Java o de Python.

### Evento esperado
```json
{
  "tipoEvento": "orden.creada",
  "ordenId": 1,
  "total": 150.0,
  "estado": "PENDIENTE",
  "origen": "java",
  "timestamp": 1710000000
}
```

Los campos `origen` y `timestamp` son opcionales.

In [5]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

schema = StructType([
    StructField("tipoEvento", StringType(), True),
    StructField("ordenId", LongType(), True),
    StructField("total", DoubleType(), True),
    StructField("estado", StringType(), True),
    StructField("origen", StringType(), True),
    StructField("timestamp", LongType(), True)
])

## Paso 4. Parsear el JSON

In [6]:
df_value = df_kafka.selectExpr("CAST(value AS STRING) AS value")

df_parsed = df_value.select(
    from_json(col("value"), schema).alias("data")
).select("data.*")

In [7]:
df_parsed.printSchema()

root
 |-- tipoEvento: string (nullable = true)
 |-- ordenId: long (nullable = true)
 |-- total: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)



## Paso 5. Validar registros

Filtramos mensajes que sí tengan al menos el contrato básico:
- `tipoEvento`
- `ordenId`
- `total`
- `estado`

In [8]:
df_validado = df_parsed.filter(
    col("tipoEvento").isNotNull() &
    col("ordenId").isNotNull() &
    col("total").isNotNull() &
    col("estado").isNotNull()
)

## Paso 6. Transformación de ejemplo

En este caso, solo dejamos órdenes con `total > 100`.

In [9]:
df_transformed = df_validado.filter(col("total") > 0)

## Paso 7. Salida a consola

In [10]:
# Momento 1: Consola debug visual
query_console = df_transformed.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()

#query_console.awaitTermination()

26/04/16 17:01:08 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-bc086c02-2622-4e38-b094-6b5d3ff4b333. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/16 17:01:08 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+------------+-------+------+---------+------+---------+
|tipoEvento  |ordenId|total |estado   |origen|timestamp|
+------------+-------+------+---------+------+---------+
|orden.creada|15     |0.1   |PENDIENTE|NULL  |NULL     |
|orden.creada|16     |10.1  |PENDIENTE|NULL  |NULL     |
|orden.creada|17     |10.1  |PENDIENTE|NULL  |NULL     |
|orden.creada|18     |200.1 |PENDIENTE|NULL  |NULL     |
|orden.creada|19     |2500.1|PENDIENTE|NULL  |NULL     |
+------------+-------+------+---------+------+---------+



## Detener la consulta
Si necesitas detener manualmente la ejecución, puedes usar:

In [15]:
#query_console.stop()

26/04/16 01:42:26 WARN DAGScheduler: Failed to cancel job group 37b14aa9-5ef0-4f51-989e-9f99a5e6e58f. Cannot find active jobs for it.
26/04/16 01:42:26 WARN DAGScheduler: Failed to cancel job group 37b14aa9-5ef0-4f51-989e-9f99a5e6e58f. Cannot find active jobs for it.


## Ejemplos de productores compatibles

### Java
```java
@NoArgsConstructor
@AllArgsConstructor
public class EventoOrden {
    private String tipoEvento;
    private Long ordenId;
    private Double total;
    private String estado;
}
```

### Python
```python
from kafka import KafkaProducer
import json, time, random

producer = KafkaProducer(
    bootstrap_servers='ec-kafka:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

while True:
    data = {
        "tipoEvento": "orden.creada",
        "ordenId": random.randint(1, 1000),
        "total": random.randint(50, 1500),
        "estado": "PENDIENTE",
        "origen": "python",
        "timestamp": int(time.time())
    }

    producer.send("orden-eventos", value=data)
    print("Enviado:", data)
    time.sleep(2)
```

In [11]:
# Momento 2: persistencia Arrancar stream a Parquet o Salida a Parquet

query_parquet = df_transformed.writeStream \
    .format("parquet") \
    .option("path", "data/orden_eventos_parquet") \
    .option("checkpointLocation", "checkpoint/orden-eventos") \
    .outputMode("append") \
    .start()

26/04/16 17:01:22 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
[Stage 1:>                                                          (0 + 1) / 1]

In [23]:
#Esperar micro-batch
import time
time.sleep(10)



In [15]:
#Leer lo guardado
df_final = spark.read.parquet("data/orden_eventos_parquet")

df_final.show(truncate=False)

+------------+-------+------+---------+------+---------+
|tipoEvento  |ordenId|total |estado   |origen|timestamp|
+------------+-------+------+---------+------+---------+
|orden.creada|18     |200.1 |PENDIENTE|NULL  |NULL     |
|orden.creada|19     |2500.1|PENDIENTE|NULL  |NULL     |
|orden.creada|16     |10.1  |PENDIENTE|NULL  |NULL     |
|orden.creada|17     |10.1  |PENDIENTE|NULL  |NULL     |
|orden.creada|15     |0.1   |PENDIENTE|NULL  |NULL     |
+------------+-------+------+---------+------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
+----------+-------+-----+------+------+---------+
|tipoEvento|ordenId|total|estado|origen|timestamp|
+----------+-------+-----+------+------+---------+
+----------+-------+-----+------+------+---------+



-------------------------------------------
Batch: 2
-------------------------------------------
+----------+-------+-----+------+------+---------+
|tipoEvento|ordenId|total|estado|origen|timestamp|
+----------+-------+-----+------+------+---------+
+----------+-------+-----+------+------+---------+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------+-------+------+---------+------+---------+
|tipoEvento  |ordenId|total |estado   |origen|timestamp|
+------------+-------+------+---------+------+---------+
|orden.creada|19     |2500.1|PENDIENTE|NULL  |NULL     |
+------------+-------+------+---------+------+---------+



In [13]:
df_final.printSchema()

root
 |-- tipoEvento: string (nullable = true)
 |-- ordenId: long (nullable = true)
 |-- total: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)



In [14]:
df_final.select("tipoEvento", "ordenId", "total", "estado").show()

+------------+-------+-----+---------+
|  tipoEvento|ordenId|total|   estado|
+------------+-------+-----+---------+
|orden.creada|     16| 10.1|PENDIENTE|
|orden.creada|     17| 10.1|PENDIENTE|
|orden.creada|     15|  0.1|PENDIENTE|
+------------+-------+-----+---------+



In [ ]:
#Detener stream cuando ya no lo necesites, Detener ambas
#query_console.stop()
#query_parquet.stop()